In [1]:
from analyses.spike_rate import compute_mean_spike_rate_for_cells, compute_mean_spike_rate_for_windows
from sklearn.cross_decomposition import PLSRegression
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# PLS
windows_df = pd.read_pickle('/home/connorlab/Documents/GitHub/Julie/Cortana/analysis_cache/Zombies_significant_windows_pANOVAorGLM_passed.pkl')
spike_df = compute_mean_spike_rate_for_windows(windows_df)

Computing windowed mean spike rate: 100%|██████████| 525/525 [00:08<00:00, 65.54it/s]


In [6]:
# Load behavioral matrix
parent_dir = "/home/connorlab/Documents/GitHub/Julie/social_data/zombies_social_data/"
# Load data
aff_df = pd.read_excel(parent_dir + "zombies_feature_df_affiliation.xlsx", index_col=0)
sub_df = pd.read_excel(parent_dir + "zombies_feature_df_submission.xlsx", index_col=0)
ago_df = pd.read_excel(parent_dir + "zombies_feature_df_agonism.xlsx", index_col=0)

In [32]:
# Load matrix
# 함수: "Behavior Towards 94B" → "94B"
def clean_col(colname):
    return colname.split()[-1]  # 마지막 단어만 추출

# 컬럼 이름 정리
aff_df.columns = [clean_col(c) for c in aff_df.columns]
sub_df.columns = [clean_col(c) for c in sub_df.columns]
ago_df.columns = [clean_col(c) for c in ago_df.columns]

aff_cols = aff_df.columns
sub_cols = sub_df.columns
ago_cols = ago_df.columns
# Prefix to indicate behavior type
aff_features = [f'Aff_{c}' for c in aff_cols]
sub_features = [f'Sub_{c}' for c in sub_cols]
ago_features = [f'Ago_{c}' for c in ago_cols]

# Combine into full feature list (matches column order of X)
feature_names = aff_features + sub_features + ago_features


# 공통 monkey 추출
monkeys = aff_df.index.intersection(sub_df.index).intersection(ago_df.index)
stimulus_monkeys= monkeys.drop("81G")
stimulus_monkeys

Index(['7124', '69X', '72X', '94B', '110E', '67G', '143H', '87J', '151J'], dtype='object', name='Focal Name')

In [20]:
# Build behavior vector per monkey (row-wise vector from each matrix)
# Remove 81G from monkey list
monkeys_no81G = [m for m in monkeys if m != '81G']

# Subset behavior matrix
X = []
for m in monkeys_no81G:
    v = np.concatenate([
        aff_df.loc[m].values,
        sub_df.loc[m].values,
        ago_df.loc[m].values
    ])
    X.append(v)
X = np.array(X)
X

array([[ 0, 19,  9, 38, 84, 27, 13, 14,  4,  9,  0,  0,  0,  1,  0,  1,
         0,  0,  2,  0,  0,  4,  2,  7,  4,  7,  3,  9,  3,  2],
       [15,  0,  9, 41,  4, 13, 19, 71, 12,  0,  4,  0,  0, 13,  0,  0,
         0,  2,  1,  0,  0,  0,  0,  2,  1,  1,  8,  4,  2,  0],
       [18, 10,  0, 21,  7, 17, 49, 18,  3,  1,  9,  2,  0, 10,  0,  0,
         4,  3,  0,  0,  0,  1,  0,  0,  0,  7,  0,  1,  0,  2],
       [38, 43, 24,  0, 18,  6, 31, 26, 29,  4,  7,  0,  2,  0,  0,  0,
         0,  1,  6,  0,  1,  8,  0,  0,  9,  5,  6, 10, 11,  2],
       [90,  3,  8,  8,  0, 22,  8,  9,  1, 18, 11,  8,  3, 15,  0,  2,
         4,  2,  4, 18,  0,  0,  0,  0,  0,  1,  0,  1,  3, 10],
       [23, 12, 17,  1, 23,  0, 23, 16,  3, 10, 90, 16, 14, 43,  0,  0,
        10,  7,  9,  6,  0,  0,  0,  0,  2,  0,  0,  0,  3,  1],
       [11, 70, 18, 17,  8, 17, 33,  0,  5,  3, 41,  6,  1, 28,  0,  0,
         2,  0,  5,  0,  0,  1,  1,  0,  1,  0,  7,  0,  0,  1],
       [ 3,  6,  4, 31,  2,  1,  2,  4,  

In [21]:
# Build Y (still z-scored across monkeys per neuron for comparability)
filtered_df = spike_df[(spike_df['MonkeyGroup'] == 'Zombies')]
filtered_df

,NeuronID,MonkeyName,MonkeyGroup,WindowStart_ms,WindowEnd_ms,MeanSpikeRate
4,AMG_2023-09-26_1_Channel.C_014_Unit 1,110E,Zombies,150.0,500.0,56.507937
6,AMG_2023-09-26_1_Channel.C_014_Unit 1,143H,Zombies,150.0,500.0,64.571429
9,AMG_2023-09-26_1_Channel.C_014_Unit 1,151J,Zombies,150.0,500.0,66.349206
22,AMG_2023-09-26_1_Channel.C_014_Unit 1,67G,Zombies,150.0,500.0,60.000000
26,AMG_2023-09-26_1_Channel.C_014_Unit 1,69X,Zombies,150.0,500.0,66.349206
...,...,...,...,...,...,...
19425,Unknown_2023-12-18_3_Channel.C_029_Unit 1,69X,Zombies,50.0,350.0,8.000000
19427,Unknown_2023-12-18_3_Channel.C_029_Unit 1,7124,Zombies,50.0,350.0,8.333333
19428,Unknown_2023-12-18_3_Channel.C_029_Unit 1,72X,Zombies,50.0,350.0,10.000000
19431,Unknown_2023-12-18_3_Channel.C_029_Unit 1,87J,Zombies,50.0,350.0,7.333333


In [22]:
value_table = filtered_df.groupby(['NeuronID', 'MonkeyName'])['MeanSpikeRate'].mean().unstack()
value_table = value_table.loc[:, stimulus_monkeys]
# Transpose for PLS (rows = monkeys, cols = neural features)
from scipy.stats import zscore
Y_df = pd.DataFrame(zscore(value_table.T, axis=0), index=stimulus_monkeys)
# Y_df = pd.DataFrame(value_table.T, index=stimulus_monkeys)
print(Y_df)

NeuronID    AMG_2023-09-26_1_Channel.C_014_Unit 1  \
Focal Name                                          
7124                                     1.425822   
69X                                      0.768670   
72X                                     -1.132641   
94B                                     -2.010778   
110E                                    -0.375809   
67G                                      0.030297   
143H                                     0.561926   
87J                                     -0.036157   
151J                                     0.768670   

NeuronID    AMG_2023-09-26_1_Channel.C_018_Unit 1  \
Focal Name                                          
7124                                     0.743919   
69X                                      0.656071   
72X                                      1.534551   
94B                                     -0.724397   
110E                                    -0.661649   
67G                                     -1.62

In [23]:
# Run PLS
from sklearn.preprocessing import StandardScaler

X_scaled = X  # no z-score, raw values
Y_scaled = StandardScaler().fit_transform(Y_df)  # standardize neural space

pls = PLSRegression(n_components=3)
pls.fit(X_scaled, Y_scaled)

PLSRegression(n_components=3)

In [24]:
from mpl_toolkits.mplot3d import Axes3D  # Required for 3D plotting

scores = pls.x_scores_  # shape: (9, 3)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(scores[:, 0], scores[:, 1], scores[:, 2], s=100)

for i, name in enumerate(stimulus_monkeys):
    ax.text(scores[i, 0] + 0.02, scores[i, 1], scores[i, 2], name, fontsize=9)

ax.set_xlabel('PLS Component 1')
ax.set_ylabel('PLS Component 2')
ax.set_zlabel('PLS Component 3')
ax.set_title('Stimulus Monkeys in 3D PLS Component Space')

plt.tight_layout()
plt.show()

In [33]:
import seaborn as sns

weights_matrix = pls.x_weights_  # shape: (num_features, num_components)
sns.heatmap(weights_matrix,
            xticklabels=[f'Comp {i+1}' for i in range(weights_matrix.shape[1])],
            yticklabels=feature_names,
            cmap='coolwarm', center=0)

plt.title('Feature Weights Across PLS Components')
plt.xlabel('PLS Components')
plt.ylabel('Features')
plt.tight_layout()
plt.show()